# Conditioning Through Singular Values

Chapter 8: Singular Value Decomposition (SVD) · *Mathematics for AI and Machine Learning* by Fuheng Wu

Inspect condition number and sensitivity as smallest singular value changes.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-8/singular-conditioning/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)

# The demo's default sliders: largest and smallest singular value, perturbation size
s_max, s_min, delta = 4.0, 0.25, 0.05
kappa = s_max / s_min
print(f"condition number kappa = s_max / s_min = {kappa}")
print(f"worst-case relative change in x: about kappa * delta = {kappa * delta}")

# ── A 2x2 system with these singular values ─────────────────────────────────
U, _ = torch.linalg.qr(torch.randn(2, 2, dtype=torch.float64))
V, _ = torch.linalg.qr(torch.randn(2, 2, dtype=torch.float64))
A = U @ torch.diag(torch.tensor([s_max, s_min], dtype=torch.float64)) @ V.T
print("torch.linalg.cond(A) =", torch.linalg.cond(A).item())

# Worst case: b along the top output direction, the error along the weakest one
b = U[:, 0] * s_max                      # so x = v1, ||x|| = 1
db = delta * (b.norm()) * U[:, 1]        # relative size delta, aimed at u2
x = torch.linalg.solve(A, b)
dx = torch.linalg.solve(A, b + db) - x
print(f"\nrelative change in b = {(db.norm() / b.norm()).item():.4f}")
print(f"relative change in x = {(dx.norm() / x.norm()).item():.4f}  (= kappa * delta)")

# ── Shrinking s_min blows the amplification up ──────────────────────────────
print("\n s_min    kappa     amplification of a relative error delta")
for sm in (1.0, 0.25, 0.05, 0.01):
    print(f"{sm:6.2f}  {s_max / sm:7.1f}   {s_max / sm * delta:.3f}")

condition number kappa = s_max / s_min = 16.0
worst-case relative change in x: about kappa * delta = 0.8
torch.linalg.cond(A) = 15.999999999999996

relative change in b = 0.0500
relative change in x = 0.8000  (= kappa * delta)

 s_min    kappa     amplification of a relative error delta
  1.00      4.0   0.200
  0.25     16.0   0.800
  0.05     80.0   4.000
  0.01    400.0   20.000
